# Page to Pixel—Digitization, OCR, and Making Texts into Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/page-to-pixel/blob/main/Page-to-Pixel.ipynb)

In this notebook, we'll explore some of the different ways that the characters and words in digitized images of texts can be converted into digital formats. We'll work with one of the most commonly used Optical Character Recognition (OCR) engines, Tesseract, in its Python wrapper, `pytesseract`. We'll also test out some of the most recent Vision Language Models (VLMs), which adapt the technology of Large Language Models (LLMs) to interpret and then generate text. Feel free to upload your own images and test out the ability of the different models to correctly extract the text. 

> **Important Step**

Before running anything: switch to a GPU runtime. We're using some AI models in this notebook and running them "locally" (which is to say that we're running them in the notebook itself). In order to do this, we're going to need GPUs or Graphical Processing Units instead of the standard CPUs or Central Processing Units. 

By default, Colab starts you on a CPU-only runtime, so switch it manually:

1. Click **Runtime** in the menu bar at the top of the page.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, choose **T4 GPU**.
4. Click **Save**.

Colab will restart your session (this is normal). Once it's back, run the cells from the top.

In [ ]:
""" If your're working in Colab, you can execute this cell to bring everything, 
including the sample data, over from the GitHub repository so that you can have 
it all in one place"""
!git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/page-to-pixel.git
%cd page-to-pixel

Cloning into 'page-to-pixel'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 18 (delta 0), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 23.97 MiB | 31.27 MiB/s, done.
/Users/bradykrien/Workspace/text-to-data_workshops/page-to-pixel/page-to-pixel


In [ ]:
# Packages to install if necessary
!apt-get install -y tesseract-ocr # Tesseract's underlying engine (pytesseract is just a Python wrapper around this)
!pip install pytesseract # main OCR package
!pip install Pillow #Python Image Library is the main image package
!pip install pandas # needed for pytesseract's DATAFRAME output
!pip install -q -U transformers accelerate torchvision # ML and VLM facilitation packages (-U for newer architectures like PaddleOCR-VL and Qwen3.5; torchvision is needed by the image/video processors)
!pip uninstall -y -q torchaudio # unused in this notebook, and its preinstalled build can end up compiled against a different CUDA version than the torch/torchvision we just upgraded, which breaks transformers imports


In [ ]:
# Packages to import
import pytesseract # Classic OCR package in Python
import torch # PyTorch is a deep learning framework
from PIL import Image # Python image processing library
from pytesseract import Output # import Output module for reporting data
import pandas as pd # pandas is useful for representing and manipulating data
from transformers import AutoProcessor, AutoModelForImageTextToText

## Tesseract OCR Library

### Dickens

We have a page from Charles Dickens' *A Tale of Two Cities* from the HathiTrust Digital Library. This image has a single column of printed text, which makes it a good candidate for basic OCR processing. The original image can be access here: [https://babel.hathitrust.org/cgi/pt?id=hvd.32044024216053&seq=21](https://babel.hathitrust.org/cgi/pt?id=hvd.32044024216053&seq=21)

In [ ]:
# Let's see what the text it outputs looks like
dickens = Image.open('sample_data/dickens.jpg') # Open the image
print(pytesseract.image_to_string(dickens)) # Process the image and output the text

A TALE OF TWO CITIES.
IN THREE BOOKS.

BY

CHARLES DICKENS.
VOL. L

ET

BOOK THE FIRST.

RECALLED TO LIFE.

CHAPTER I.
The Period.

Iv was the best of times, it was the worst of times,
it was the age of wisdom, it was the age of foolishness,
it was the epoch of belief, it was the epoch of incre-
dulity, it was the season of Light, it was the season
of Darkness, it was the spring of hope, it was the win-
ter of despair, we had everything before us, we had
nothing before us, we were all going direct to Heaven,
we were all going direct the other way — in short, the
period was so far like the present period, that some of
its noisiest authorities insisted on its being received,
_for good or for evil, in the superlative degree of com-
parison only. * 1%

Google



### Getting confidence scores

`image_to_string` only returns plain text. To get Tesseract's per-word confidence (0-100, or -1 for non-text regions like lines), use `image_to_data`, which also gives bounding boxes for each recognized word.

In [ ]:
dickens_data = pytesseract.image_to_data(dickens, output_type=Output.DATAFRAME) # Let's take a look at the data 

# Drop rows with no recognized text (conf == -1) and preview word-level confidence
dickens_words = dickens_data[dickens_data.conf != -1][['text', 'conf', 'left', 'top', 'width', 'height']]
dickens_words.head(20)

In [ ]:
# A single overall confidence score for the page
print(f"Average word confidence for the dickens text: {dickens_words.conf.mean():.1f}")

### The Gettysburg Address

Our next image is significantly more complicated. It's a handwritten draft of Lincoln's "Gettysburg Address." Handwriting, even relatively neat and legible handwriting, is much less consistent that printed text. This image is from the Library of Congress and can be accessed at: [https://www.loc.gov/resource/mal.4356600/?st=text&r=-0.066,-0.288,1.111,1.432,0](https://www.loc.gov/resource/mal.4356600/?st=text&r=-0.066,-0.288,1.111,1.432,0)

In [10]:
# Again, we'll start with the text output
gettysburg = Image.open('sample_data/gettysburg.jpg') # Open the image
print(pytesseract.image_to_string(gettysburg)) # Process the image and output the text

ruv ecme anev seven yeors age Cry forbes
bange frk, pow KA Coutenend, a henwr ration, Cont
Cewees tw VberS” an clecteelcn 6 Be frgftentins
VOTE OLE! Piven, Gags Cie CeO agrck,

Now pres Be Prgepeed pe yore Cink way Ge:
“ey whether LKaD ator oy Ang Malach, bo Conceunal,
Prov fo clectes€ev Con beg preter, Ne aw Mer
PS re a ped batho feted fe ie

meds: eee Clecthitvets a polos Ge ad ite fica

Prev fvcfier AAEO bres pho Cs ale ee,

Pe ew ae Farge br 20 We Caw [rut clectesta—
We Cen (hoC Coiecie Wwe Caw hod halen ges
Y acenanie tke Qs Trew fatting
poor
, mia dics as Cokzgen geo tO fe aleve cu, ower
be acto or eCOrzee, DE wWorkev nr £6 Le, i
(ov fog pemmeintey t\hal Gr Hay. Keres Via i

vey




In [ ]:
# Let's take a look at the data and, in particular, the confidence:
gettysburg_data = pytesseract.image_to_data(gettysburg, output_type=Output.DATAFRAME) # Let's take a look at the data 

# Drop rows with no recognized text (conf == -1) and preview word-level confidence
gettysburg_words = gettysburg_data[gettysburg_data.conf != -1][['text', 'conf', 'left', 'top', 'width', 'height']]
gettysburg_words.head(20)

In [ ]:
# A single overall confidence score for the page
print(f"Average word confidence for the gettysburg text: {gettysburg_words.conf.mean():.1f}")

## Vision Language Models

While Vision Language Models (VLMs) perform a similar task to traditional Optical Character Recognition tools like Tesseract, they go about it in a very different way. They are partly interpretive but also partly generative, which means that, when applied to a text they can't correctly process, they are much less likely to output meaningless strings of characters but much more likely to output coherent but incorrect information. 

In [ ]:
# Both VLMs below will use whatever accelerator is available
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# T4 GPUs (Colab's free tier) are Turing-generation and lack hardware-accelerated
# bfloat16, so float16 runs faster there; Ampere+ GPUs and Apple Silicon (MPS)
# handle bfloat16 natively.
if device == "cuda":
    model_dtype = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
elif device == "mps":
    model_dtype = "bfloat16"
else:
    model_dtype = "float32"

print(f"Using device: {device} ({model_dtype})")
if device == "cpu":
    print("No GPU detected. In Colab, go to Runtime → Change runtime type → T4 GPU, then rerun this cell. The VLM cells below will be very slow on CPU.")

### General Vision Language Model 
Use the Qwen model here

[Qwen3.5](https://huggingface.co/Qwen/Qwen3.5-0.8B) is an open-weight model created by the Chinese company, Alibaba. It's trained on interleaved text, image, and video bit it's a general-purpose chat model, not an OCR specialist, so we prompt it conversationally and give it our handwritten Gettysburg Address image.

In [ ]:
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-0.8B")
qwen_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    dtype=model_dtype,
    attn_implementation="sdpa", # avoids the eager attention fallback, which is far more memory-hungry
).to(device)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": gettysburg},
            {"type": "text", "text": "Transcribe all of the text in this image."},
        ],
    }
]

qwen_inputs = qwen_processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(qwen_model.device)

qwen_outputs = qwen_model.generate(**qwen_inputs, max_new_tokens=512)
qwen_result = qwen_processor.decode(
    qwen_outputs[0][qwen_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(qwen_result)

### OCR Vision Language Model
Use the GLM-OCR model here.

[GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) is a compact (0.9B parameter) document-parsing VLM from Zhipu AI (Z.ai), currently ranked #1 on the OmniDocBench V1.5 leaderboard. Like PaddleOCR-VL, it's a specialist rather than a general chat model, but it's driven the same way we prompted Qwen3.5 above — through a chat template — with a short task instruction such as `"OCR:"`.

In [ ]:
glm_processor = AutoProcessor.from_pretrained("zai-org/GLM-OCR")
glm_model = AutoModelForImageTextToText.from_pretrained(
    "zai-org/GLM-OCR",
    dtype=model_dtype,
    attn_implementation="sdpa", # avoids the eager attention fallback, which is far more memory-hungry
).to(device)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": gettysburg},
            {"type": "text", "text": "OCR:"},
        ],
    }
]

glm_inputs = glm_processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(glm_model.device)
glm_inputs.pop("token_type_ids", None)  # GLM-OCR's processor adds this, but generate() doesn't accept it

glm_outputs = glm_model.generate(**glm_inputs, max_new_tokens=512)
glm_result = glm_processor.decode(
    glm_outputs[0][glm_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(glm_result)

# GLM-OCR is trained to emit layout-aware Markdown (headers, lists, tables) for structured pages,
# so we save its output as a .md file to preserve any formatting it produced
with open('glm_result.md', 'w') as f:
    f.write(glm_result)

## Try Your Own Image

Feel free to test everything above on an image of your own — a photo of a book page, a handwritten note, anything with text in it. Run the cell below to upload a file from your computer.

A couple of things to keep in mind:
- The upload only lasts for this Colab session — it isn't saved anywhere once you disconnect.
- This is a shared classroom machine, so avoid uploading anything sensitive or personal.

In [ ]:
from google.colab import files

uploaded = files.upload()
my_image_path = list(uploaded.keys())[0]
my_image = Image.open(my_image_path)
my_image # Displays the image inline so you can confirm the right file loaded

### Tesseract

In [ ]:
print(pytesseract.image_to_string(my_image))

### Vision Language Models

The `qwen_model` and `glm_model` from above are still loaded in memory, so trying them on your image doesn't require re-downloading anything — just build a new prompt with your image. Here's GLM-OCR again, this time on your upload:

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": my_image},
            {"type": "text", "text": "OCR:"},
        ],
    }
]

my_inputs = glm_processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(glm_model.device)
my_inputs.pop("token_type_ids", None)

my_outputs = glm_model.generate(**my_inputs, max_new_tokens=512)
my_result = glm_processor.decode(
    my_outputs[0][my_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(my_result)

Want to try Qwen3.5 instead? Copy the cell above, swap `glm_processor`/`glm_model` for `qwen_processor`/`qwen_model`, and change the prompt to something like `"Transcribe all of the text in this image."`